# Contingency screening: which outages to check first

A full AC N-1 analysis runs one power flow per outage. On large grids it helps to rank the outages from the base case
first, so the AC analysis can start with the ones most likely to overload other branches. This tutorial ranks every
line outage of `case_illinois200` and then checks each ranking against an AC power flow for every outage. Loads and
generation are scaled up by 12% so that some outages do overload lines or transformers.

The rankings compared are:

* the pre-outage loading and the pre-outage active power flow of the outaged line,
* DC LODF screening with `pandapower.sensitivity.run_lodf`, which predicts every post-outage flow from the line
  outage distribution factors and adds up the predicted new overloads,
* a structural criticality index (CEI) that needs no flow model. It combines the line's centrality in the grid, the
  entropy of its daily utilization and its voltage level.

Each ranking is scored by how early it finds the outages that cause a new overload.

In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

import pandapower as pp
import pandapower.networks as pn
import pandapower.topology as top
from pandapower.sensitivity import run_lodf

net = pn.case_illinois200()
for table, cols in (("load", ["p_mw", "q_mvar"]), ("gen", ["p_mw"]), ("sgen", ["p_mw"])):
    net[table][cols] *= 1.12  # stress the grid a little
pp.runpp(net)
base = net.res_line.copy()
base_trafo = net.res_trafo.copy()
lines = net.line.index[net.line.in_service].to_numpy()
rating_mva = math.sqrt(3) * net.bus.vn_kv.loc[net.line.from_bus].to_numpy() * net.line.max_i_ka.to_numpy()
rating_mva = pd.Series(rating_mva, index=net.line.index)
print(f"{len(net.bus)} buses, {len(lines)} lines, highest base-case loading {base.loading_percent.max():.0f}%")

## Ground truth: AC power flow for every line outage

An outage counts as severe if it pushes any other line or any transformer above both 100% and its own pre-outage
loading (by more than 0.1 percentage points), or if the power flow does not converge. Outages that cut buses off the
grid (radial lines) are left out, because a topology check already finds them.

In [ ]:
def new_overload(net, out_line):
    excess = pd.concat(
        [
            net.res_line.loading_percent.drop(out_line) - np.maximum(100.0, base.loading_percent.drop(out_line)),
            net.res_trafo.loading_percent - np.maximum(100.0, base_trafo.loading_percent),
        ]
    )
    return excess.clip(lower=0).sum() / 100.0, bool((excess > 0.1).any())


rows = []
for k in lines:
    net.line.at[k, "in_service"] = False
    try:
        pp.runpp(net, init="dc")
        islanded = bool(net.res_bus.vm_pu.isna().any())
        osi, severe = new_overload(net, k)
    except pp.LoadflowNotConverged:
        islanded, osi, severe = False, np.inf, True
    finally:
        net.line.at[k, "in_service"] = True
    rows.append({"line": k, "islanded": islanded, "osi": osi, "severe": severe})
truth = pd.DataFrame(rows).set_index("line")
truth = truth[~truth.islanded]
pp.runpp(net)  # restore the base-case results
print(f"{len(truth)} non-radial outages, {truth.severe.sum()} severe, {np.isinf(truth.osi).sum()} not converged")

## Rankings from the base case: loading, flow and DC LODF screening

`run_lodf` returns the LODF matrices: the share of an outaged line's flow that moves onto each other line and
each transformer. With the base-case flows this predicts every post-outage flow without another power flow.

In [ ]:
lodf_all = run_lodf(net, "line", distributed_slack=False)
trafos = net.trafo.index[net.trafo.in_service]
lodf = pd.concat(
    [
        lodf_all[("line", "line")].reindex(index=lines, columns=truth.index),
        lodf_all[("trafo", "line")].reindex(index=trafos, columns=truth.index),
    ]
)
p0 = np.r_[base.p_from_mw.loc[lines].to_numpy(), base_trafo.p_hv_mw.loc[trafos].to_numpy()]
rate = np.r_[rating_mva.loc[lines].to_numpy(), net.trafo.sn_mva.loc[trafos].to_numpy()]
lam0 = np.abs(p0) / rate * 100
pos = pd.Series(np.arange(len(lines)), index=lines)

screen = {}
for k in truth.index:
    lam = np.abs(p0 + lodf[k].to_numpy() * p0[pos[k]]) / rate * 100
    keep = np.arange(len(p0)) != pos[k]
    over = np.clip(lam[keep] - np.maximum(100, lam0[keep]), 0, None).sum() / 100
    worst = np.max(lam[keep] / np.maximum(100, lam0[keep]))
    screen[k] = (over, worst)
screen = pd.DataFrame(screen, index=["over", "worst"]).T

scores = pd.DataFrame(index=truth.index)
scores["loading"] = base.loading_percent.loc[truth.index]
scores["flow"] = base.p_from_mw.loc[truth.index].abs()
# rank by predicted new overload, then by the worst predicted loading ratio
scores["lodf"] = screen.over.rank(method="dense") + screen.worst.rank(method="dense") / (len(screen) + 1)

## A structural criticality index

The index is a weighted sum of three parts, with weights 0.40, 0.35 and 0.25:

* C, a composite centrality (0.35 PageRank + 0.30 betweenness + 0.20 degree + 0.15 closeness, scaled to a maximum
  of 1) computed on the line graph, where every branch is a node,
* H, the Shannon entropy of the line's utilization over a 24-hour day (10 bins), here from the IEEE RTS winter
  weekday load shape with a little noise per load,
* R, the line's nominal voltage divided by the highest in the grid, standing in for a criticality tier an operator
  would declare.

In [ ]:
mg = top.create_nxgraph(net, multi=True, calc_branch_impedances=False)
g = nx.MultiGraph((u, v, k) for u, v, k in mg.edges(keys=True) if u != v)
lg = nx.line_graph(g)
node = {n[2]: n for n in lg.nodes}
pr, bc = nx.pagerank(lg), nx.betweenness_centrality(lg)
dc, cc = nx.degree_centrality(lg), nx.closeness_centrality(lg)


def composite(k):
    n = node[("line", k)]
    return 0.35 * pr[n] + 0.30 * bc[n] + 0.20 * dc[n] + 0.15 * cc[n]


C = pd.Series({k: composite(k) for k in lines})
C = C / C.max()

# IEEE RTS winter weekday load, % of the daily peak
rts_day = [67, 63, 60, 59, 59, 60, 74, 86, 95, 96, 96, 95, 95, 95, 93, 94, 99, 100, 100, 96, 91, 83, 73, 63]
shape = np.array(rts_day) / 100
rng = np.random.default_rng(0)
p_load, q_load, p_gen = net.load.p_mw.copy(), net.load.q_mvar.copy(), net.gen.p_mw.copy()
history = []
for s in shape:
    noise = rng.uniform(0.95, 1.05, len(net.load))
    net.load.p_mw, net.load.q_mvar, net.gen.p_mw = p_load * s * noise, q_load * s * noise, p_gen * s
    pp.runpp(net, init="dc")
    history.append(net.res_line.loading_percent.loc[lines].to_numpy())
net.load.p_mw, net.load.q_mvar, net.gen.p_mw = p_load, q_load, p_gen
pp.runpp(net)


def entropy(u, bins=10):
    h, _ = np.histogram(np.clip(u / 100, 0, 1), bins=bins, range=(0, 1))
    p = h[h > 0] / h.sum()
    return float(-(p * np.log2(p)).sum() / math.log2(bins))


H = pd.Series([entropy(col) for col in np.array(history).T], index=lines)
vn = net.bus.vn_kv.loc[net.line.from_bus.loc[lines]].to_numpy()
R = pd.Series(vn / vn.max(), index=lines)
scores["cei"] = (0.40 * C + 0.35 * H + 0.25 * R).loc[truth.index]
scores["cei_centrality"] = C.loc[truth.index]

## How well does each ranking find the severe outages?

*Average precision* is the area under the precision-recall curve (1 is perfect, and a random order scores the share
of severe outages). *Recall in the top 10%* is the share of severe outages found if only the top-ranked tenth is
checked with AC power flow.

In [ ]:
def average_precision(score, y):
    order = np.argsort(-score.to_numpy(), kind="stable")
    s, y = score.to_numpy()[order], y.to_numpy()[order]
    ends = np.flatnonzero(np.r_[s[1:] != s[:-1], True])  # equal scores count as one threshold
    tp = np.cumsum(y)[ends]
    precision, recall = tp / (ends + 1), tp / y.sum()
    return float(np.sum(np.diff(np.r_[0, recall]) * precision))


def recall_top(score, y, frac=0.10):
    n = math.ceil(frac * len(score))
    return float(y.loc[score.sort_values(ascending=False).index[:n]].sum() / y.sum())


y = truth.severe
table = pd.DataFrame(
    {name: [average_precision(s, y), recall_top(s, y)] for name, s in scores.items()},
    index=["average precision", "recall in top 10%"],
).T
table.loc["random order"] = [y.mean(), 0.10]
table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name, s in scores.items():
    found = y.loc[s.sort_values(ascending=False).index].cumsum().to_numpy() / y.sum()
    ax.plot(np.arange(1, len(found) + 1) / len(found), found, label=name)
ax.plot([0, 1], [0, 1], "k:", label="random order")
ax.set_xlabel("share of outages checked with AC power flow")
ax.set_ylabel("share of severe outages found")
ax.legend()
plt.show()

## Why DC screening misses outages on this grid

DC LODF screening predicts only how active power moves. For the five worst outages, look at the element that
overloads: its loading rises while its active power stays the same, because the extra current is reactive or comes
from a lower voltage. A DC method cannot see that, so it is worth checking a DC screen against AC results on your
own grid before relying on it.

In [ ]:
rows = []
for k in truth[truth.severe].sort_values("osi", ascending=False).index[:5]:
    if np.isinf(truth.at[k, "osi"]):  # the power flow for this outage did not converge above
        rows.append({"outaged line": k, "overloaded": "power flow did not converge"})
        continue
    net.line.at[k, "in_service"] = False
    try:
        pp.runpp(net, init="dc")
    finally:
        net.line.at[k, "in_service"] = True
    excess = pd.concat(
        [
            (net.res_line.loading_percent - np.maximum(100.0, base.loading_percent)).drop(k).add_prefix("line "),
            (net.res_trafo.loading_percent - np.maximum(100.0, base_trafo.loading_percent)).add_prefix("trafo "),
        ]
    )
    table, j = excess.idxmax().split()
    j = int(j)
    res, res0, p_col = (net.res_line, base, "p_from_mw") if table == "line" else (net.res_trafo, base_trafo, "p_hv_mw")
    rows.append(
        {
            "outaged line": k,
            "overloaded": f"{table} {j}",
            "loading before %": res0.at[j, "loading_percent"],
            "loading after %": res.at[j, "loading_percent"],
            "P before MW": res0.at[j, p_col],
            "P after MW": res.at[j, p_col],
        }
    )
pp.runpp(net)
pd.DataFrame(rows).round(1)

## The same comparison on five grids

The comparison above was repeated on five test grids (case118 with ratings set from its base-case flows,
case_illinois200, case1354pegase, case1888rte and case2869pegase), each at ten load levels from 80% to 116% of the
nominal case. DC LODF screening found the severe outages almost perfectly on case118, case1354pegase and case2869pegase (average precision
0.99), where overloads come from active power moving to other branches. On case_illinois200 (0.19) and case1888rte (0.31) it was weak for the
reason shown above, and ranking by pre-outage flow did better (0.61 and 0.39). The structural index scored above edge
betweenness and at or below pre-outage flow on every grid, and adding it to the LODF ranking improved screening on
only 2 of the 5 grids.